In [2]:
import pandas as pd
from sklearn.preprocessing import minmax_scale
import os

In [3]:
df_future = pd.read_csv("../src/data/files/FUTURE_CAMPAIGN_PE_OFFERS.csv")

In [4]:
# For future campaigns
df_future_filtered = df_future[df_future["CODCUC"] != "XXXXXXXXX"]

print(df_future_filtered.shape)

(44125, 26)


In [5]:
# For future campaigns
df_future_filtered = df_future_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1)  # Remove "Column2"
print(df_future_filtered.shape)

(44125, 25)


In [6]:
unique_occurrences = df_future_filtered["COD_PERIODO"].value_counts()

print("Unique occurrences:")
print(unique_occurrences)

Unique occurrences:
COD_PERIODO
202501    24918
202502    16509
202503     2698
Name: count, dtype: int64


In [7]:
df_future_filtered["Composite_key"] = df_future_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)

In [8]:
# Divide the DataFrame based on unique values in the 'COD_PERIODO' column
df_futures = {campaign_id: df_subset for campaign_id, df_subset in df_future_filtered.groupby('COD_PERIODO')}

In [9]:
# Assuming dfs contains the smaller DataFrames (subsets) from the previous steps
# Example of transformations on each sub_df
for i, (campaign_id, sub_df) in enumerate(df_futures.items(), 1):
    
    # Perform the groupby and aggregation
    sub_df_concatenated = sub_df.groupby("ID_OFERTA")[["Composite_key"]].agg(
        {'Composite_key': lambda x: '|'.join(x)}
    ).reset_index()
    
    # Save the result to a CSV file with a dynamic name
    file_name = f"{campaign_id}.csv"
    sub_df_concatenated.to_csv(file_name, index=False)
    
    print(f"Saved {file_name}")

Saved 202501.csv
Saved 202502.csv
Saved 202503.csv


In [10]:
# Calculate the percentage of offers present in the next campaign
def calculate_key_overlap_percentage(csv1_path, csv2_path):

    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)
    
    # Clean the composite keys (remove whitespace and convert to lowercase)
    df1['Composite_key'] = df1['Composite_key'].str.strip().str.lower()
    df2['Composite_key'] = df2['Composite_key'].str.strip().str.lower()
    
    # Get unique composite keys from both DataFrames
    keys_in_csv1 = set(df1['Composite_key'].unique())
    keys_in_csv2 = set(df2['Composite_key'].unique())
    
    # Find overlapping keys
    common_keys = keys_in_csv1.intersection(keys_in_csv2)
    
    # Calculate percentage
    overlap_percentage = (len(common_keys) / len(keys_in_csv1)) * 100
    
    # Compile statistics
    stats = {
        'total_keys_csv1': len(keys_in_csv1),
        'total_keys_csv2': len(keys_in_csv2),
        'common_keys': len(common_keys),
        'overlap_percentage': round(overlap_percentage, 2)
    }
    
    return overlap_percentage, stats



In [11]:

csv1_path = "202501.csv"
csv2_path = "202502.csv"

csv1_value = os.path.splitext(os.path.basename(csv1_path))[0]
csv2_value = os.path.splitext(os.path.basename(csv2_path))[0]
percentage, stats = calculate_key_overlap_percentage(csv1_path, csv2_path)

print(f"\nResults:")
print(f"Total unique offers in {csv1_value}: {stats['total_keys_csv1']}")
print(f"Total unique offers in {csv2_value}: {stats['total_keys_csv2']}")
print(f"Number of common offers: {stats['common_keys']}")
print(f"Percentage of offers of {csv1_value} present in {csv2_value}: {stats['overlap_percentage']}%")


Results:
Total unique offers in 202501: 3907
Total unique offers in 202502: 2801
Number of common offers: 1064
Percentage of offers of 202501 present in 202502: 27.23%


In [12]:
csv1_path = "202502.csv"
csv2_path = "202503.csv"

csv1_value = os.path.splitext(os.path.basename(csv1_path))[0]
csv2_value = os.path.splitext(os.path.basename(csv2_path))[0]
percentage, stats = calculate_key_overlap_percentage(csv1_path, csv2_path)

print(f"\nResults:")
print(f"Total unique offers in {csv1_value}: {stats['total_keys_csv1']}")
print(f"Total unique offers in {csv2_value}: {stats['total_keys_csv2']}")
print(f"Number of common offers: {stats['common_keys']}")
print(f"Percentage of offers of {csv1_value} present in {csv2_value}: {stats['overlap_percentage']}%")


Results:
Total unique offers in 202502: 2801
Total unique offers in 202503: 1031
Number of common offers: 560
Percentage of offers of 202502 present in 202503: 19.99%
